# Prepping Data Challenge - Week 24

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
manager_salaries_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_24_Managers.xlsx', sheet_name='Tenures & Salaries')
yearly_expenses_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_24_Managers.xlsx', sheet_name='Total Expenses')

In [3]:
manager_salaries_df.head()

,Name,Gender,Store City,Store Area,Start Date,End Date,Salary
0,Pablo Prep,M,London,Stratford,2008-07-20,2017-03-02 00:00:00,40000
1,Phil Down,M,London,Oxford Street,2008-07-20,2020-06-04 00:00:00,40000
2,Beau Leon,M,Nottingham,NaN,2008-07-20,2018-09-05 00:00:00,35000
3,Tasha Board,F,Leeds,NaN,2008-07-20,Current,35000
4,Jewel Axis,M,Manchester,Trafford,2008-12-22,2022-01-07 00:00:00,37000


In [4]:
yearly_expenses_df.head()

,Year,Expenses
0,2008,2404683
1,2009,1954242
2,2010,2608345
3,2011,2546432
4,2012,3842963


## Challenges

1. Weekly manager salaries
2. Percentage of expenses

In [5]:
# Turn End Date to Numeric
CURRENT_END_DATE = '2024-06-12 00:00:00'
manager_salaries_df['End Date'] = np.where(manager_salaries_df['End Date'] == 'Current', CURRENT_END_DATE, manager_salaries_df['End Date'])
manager_salaries_df['End Date'] = pd.to_datetime(manager_salaries_df['End Date'])

In [6]:
manager_salaries_df.head()

,Name,Gender,Store City,Store Area,Start Date,End Date,Salary
0,Pablo Prep,M,London,Stratford,2008-07-20,2017-03-02,40000
1,Phil Down,M,London,Oxford Street,2008-07-20,2020-06-04,40000
2,Beau Leon,M,Nottingham,NaN,2008-07-20,2018-09-05,35000
3,Tasha Board,F,Leeds,NaN,2008-07-20,2024-06-12,35000
4,Jewel Axis,M,Manchester,Trafford,2008-12-22,2022-01-07,37000


### Create a weekly salary payment DataFrame

In [7]:
manager_salaries_df['Weekly Salary'] = (manager_salaries_df['Salary'] / 52).round(2)

In [8]:
weekly_salaries_by_manager = []
for manager in manager_salaries_df['Name'].unique():
    # Filter for the specific manager
    manager_data = manager_salaries_df[manager_salaries_df['Name'] == manager]

    weekly_salary = pd.DataFrame()
    
    weekly_salary['Week'] = pd.date_range(start=manager_data['Start Date'].values[0], end=manager_data['End Date'].values[0], freq='W')
    weekly_salary['Name'] = manager
    weekly_salary['Salary'] = manager_data['Weekly Salary'].values[0]
    weekly_salaries_by_manager.append(weekly_salary)

In [9]:
weekly_salary_df = pd.concat(weekly_salaries_by_manager)

In [10]:
weekly_salary_payments = pd.DataFrame(weekly_salary_df.groupby('Week')['Salary'].sum())

In [11]:
weekly_salary_payments.head()

,Salary
Week,
2008-07-20,2884.62
2008-07-27,2884.62
2008-08-03,2884.62
2008-08-10,2884.62
2008-08-17,2884.62


### Calculate the yearly percentage spent on manager salaries

In [12]:
weekly_salary_payments['Year'] = weekly_salary_payments.index.year

In [13]:
yearly_salary_payments = pd.DataFrame(weekly_salary_payments.groupby('Year')['Salary'].sum())

In [14]:
manager_expenses = yearly_salary_payments.merge(yearly_expenses_df, on='Year')
manager_expenses['Manager Salary % of Expenses'] = (manager_expenses['Salary'] / manager_expenses['Expenses']).round(3) * 100

In [15]:
manager_expenses[['Year', 'Manager Salary % of Expenses']]

,Year,Manager Salary % of Expenses
0,2008,2.9
1,2009,9.6
2,2010,8.9
3,2011,10.6
4,2012,7.3
5,2013,10.3
6,2014,10.2
7,2015,11.7
8,2016,9.4
9,2017,10.2


## Save Data

In [16]:
weekly_salary_payments.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/24_weekly_manager_expenses.csv')
manager_expenses[['Year', 'Manager Salary % of Expenses']].to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/24_manager_pct.csv')